In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ECommerce Large Scale Processing") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Session created successfully!")

Spark version: 4.0.2
Session created successfully!


In [3]:
import random
from pyspark.sql import Row
from pyspark.sql.types import *

random.seed(42)

categories = ['Electronics', 'Clothing', 'Books', 'Sports', 'Home & Kitchen']
regions = ['North', 'South', 'East', 'West', 'Central']
genders = ['Male', 'Female']

def generate_row(i):
    quantity = random.randint(1, 10)
    unit_price = round(random.uniform(5.0, 1500.0), 2)
    discount = random.choice([0.0, 0.05, 0.10, 0.15, 0.20])
    total_amount = round(quantity * unit_price * (1 - discount), 2)
    profit = round(total_amount * random.uniform(0.05, 0.40), 2)

    return Row(
        order_id=i,
        customer_id=random.randint(1, 10000),
        product_id=random.randint(1, 500),
        category=random.choice(categories),
        region=random.choice(regions),
        gender=random.choice(genders),
        year=random.randint(2021, 2024),
        month=random.randint(1, 12),
        quantity=quantity,
        unit_price=unit_price,
        discount=discount,
        total_amount=total_amount,
        profit=profit
    )

print("Generating 1 million rows...")
data = [generate_row(i) for i in range(1, 1000001)]
df = spark.createDataFrame(data)
df.cache()

print(f"Total rows: {df.count():,}")
print("Schema:")
df.printSchema()

Generating 1 million rows...
Total rows: 1,000,000
Schema:
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- year: long (nullable = true)
 |-- month: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- profit: double (nullable = true)



In [4]:
from pyspark.sql.functions import col

print(f"Before cleaning: {df.count():,} rows")

df_clean = df \
    .dropna() \
    .filter(col("total_amount") > 0) \
    .filter(col("profit") > 0) \
    .filter(col("quantity") >= 1) \
    .dropDuplicates(["order_id"])

df_clean.cache()

print(f"After cleaning: {df_clean.count():,} rows")
print(f"Removed: {df.count() - df_clean.count():,} bad records")

Before cleaning: 1,000,000 rows
After cleaning: 1,000,000 rows
Removed: 0 bad records


In [5]:
from pyspark.sql.functions import sum, avg, count, round, rank, desc
from pyspark.sql.window import Window

# 1. Revenue by Category
print("=== Revenue by Category ===")
df_clean.groupBy("category") \
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_units_sold"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(sum("profit"), 2).alias("total_profit")
    ) \
    .orderBy(desc("total_revenue")) \
    .show()

# 2. Customer Lifetime Value (Top 20)
print("=== Top 20 Customers by Lifetime Value ===")
df_clean.groupBy("customer_id") \
    .agg(
        round(sum("total_amount"), 2).alias("lifetime_value"),
        count("order_id").alias("total_orders")
    ) \
    .orderBy(desc("lifetime_value")) \
    .limit(20) \
    .show()

# 3. Monthly Revenue Trend
print("=== Monthly Revenue Trend ===")
df_clean.groupBy("year", "month") \
    .agg(round(sum("total_amount"), 2).alias("monthly_revenue")) \
    .orderBy("year", "month") \
    .show(48)

=== Revenue by Category ===
+--------------+------------+----------------+--------------+--------------+
|      category|total_orders|total_units_sold| total_revenue|  total_profit|
+--------------+------------+----------------+--------------+--------------+
|      Clothing|      199857|         1101153|7.4647530214E8|1.6763491557E8|
|Home & Kitchen|      200372|         1102315|7.4646987038E8|1.6777714438E8|
|        Sports|      199832|         1099342|7.4530854559E8|1.6747652652E8|
|   Electronics|      199877|         1099709|7.4466100593E8|1.6724247118E8|
|         Books|      200062|         1099110|7.4432537133E8|1.6758072733E8|
+--------------+------------+----------------+--------------+--------------+

=== Top 20 Customers by Lifetime Value ===
+-----------+--------------+------------+
|customer_id|lifetime_value|total_orders|
+-----------+--------------+------------+
|       4434|     587693.05|         129|
|       2107|     564403.15|         123|
|       7478|     562477.

In [6]:
from pyspark.sql.functions import sum, round, desc, col, format_number
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Fix formatting - Revenue by Category clean
print("=== Revenue by Category (Formatted) ===")
df_clean.groupBy("category") \
    .agg(
        F.count("order_id").alias("total_orders"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("profit"), 2).alias("total_profit"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value")
    ) \
    .orderBy(F.desc("total_revenue")) \
    .show(truncate=False)

# Window Function - Rank customers by spend within each region
print("=== Top 3 Customers per Region by Spend ===")
window_region = Window.partitionBy("region").orderBy(F.desc("total_amount"))

df_clean.withColumn("rank", F.rank().over(window_region)) \
    .filter(F.col("rank") <= 3) \
    .select("region", "customer_id", "total_amount", "rank") \
    .orderBy("region", "rank") \
    .show(truncate=False)

# Running Total - Cumulative revenue by year and month
print("=== Cumulative Revenue by Month ===")
monthly = df_clean.groupBy("year", "month") \
    .agg(F.round(F.sum("total_amount"), 2).alias("monthly_revenue"))

window_running = Window.orderBy("year", "month").rowsBetween(Window.unboundedPreceding, Window.currentRow)

monthly.withColumn("cumulative_revenue", F.round(F.sum("monthly_revenue").over(window_running), 2)) \
    .orderBy("year", "month") \
    .show(48, truncate=False)

=== Revenue by Category (Formatted) ===
+--------------+------------+--------------+--------------+---------------+
|category      |total_orders|total_revenue |total_profit  |avg_order_value|
+--------------+------------+--------------+--------------+---------------+
|Clothing      |199857      |7.4647530214E8|1.6763491557E8|3735.05        |
|Home & Kitchen|200372      |7.4646987038E8|1.6777714438E8|3725.42        |
|Sports        |199832      |7.4530854559E8|1.6747652652E8|3729.68        |
|Electronics   |199877      |7.4466100593E8|1.6724247118E8|3725.6         |
|Books         |200062      |7.4432537133E8|1.6758072733E8|3720.47        |
+--------------+------------+--------------+--------------+---------------+

=== Top 3 Customers per Region by Spend ===
+-------+-----------+------------+----+
|region |customer_id|total_amount|rank|
+-------+-----------+------------+----+
|Central|5292       |14997.0     |1   |
|Central|7209       |14996.7     |2   |
|Central|9731       |14991.8   

In [7]:
# Save cleaned data to Parquet
df_clean.write.mode("overwrite").parquet("/content/output/orders_clean.parquet")
print("Saved 1M rows to Parquet")

# Save category revenue to CSV
df_clean.groupBy("category") \
    .agg(
        F.count("order_id").alias("total_orders"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("profit"), 2).alias("total_profit"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value")
    ) \
    .orderBy(F.desc("total_revenue")) \
    .toPandas() \
    .to_csv("/content/output/revenue_by_category.csv", index=False)
print("Saved category revenue to CSV")

# Save top customers to CSV
df_clean.groupBy("customer_id") \
    .agg(
        F.round(F.sum("total_amount"), 2).alias("lifetime_value"),
        F.count("order_id").alias("total_orders")
    ) \
    .orderBy(F.desc("lifetime_value")) \
    .limit(100) \
    .toPandas() \
    .to_csv("/content/output/top_customers.csv", index=False)
print("Saved top customers to CSV")

print("\nAll outputs saved successfully!")

Saved 1M rows to Parquet
Saved category revenue to CSV
Saved top customers to CSV

All outputs saved successfully!
